<a href="https://colab.research.google.com/github/dinanrzki/Junior-Data-Analyst-Project/blob/main/GAYANARA_FORECASTING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'DejaVu Sans'

# =====================================================================
# 0. KONFIGURASI
# =====================================================================
DATA_DIR = '/content/gayanara-forecasting/data/'
OUT_DIR = 'output_data/'
VIZ_DIR = 'visuals/'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(VIZ_DIR, exist_ok=True)

FORECAST_HORIZON = 6  # bulan ke depan

# Palet warna brand
ROSE_GOLD = '#B76E79'
MINT = '#5FBF95'
DARK = '#3D2C2E'
LIGHT_PINK = '#FBEFF1'
GREY = '#6B6B6B'


def style_ax(ax):
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.25)


def fmt_idr(x, short=True):
    if short:
        if abs(x) >= 1e9:
            return f'Rp {x/1e9:.2f} M'
        if abs(x) >= 1e6:
            return f'Rp {x/1e6:.0f} Jt'
        return f'Rp {x:,.0f}'.replace(',', '.')
    return f'Rp {x:,.0f}'.replace(',', '.')


# =====================================================================
# Data Extraction (Added to fix FileNotFoundError)
# =====================================================================
import zipfile

zip_path = '/content/gayanara-forecasting.zip'
extract_dir = '/content/' # Extract to content, assuming data folder is inside

if not os.path.exists(DATA_DIR): # Only extract if data directory doesn't exist
    print('Extracting data from:', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print('Data extracted to:', extract_dir)
else:
    print('Data directory already exists, skipping extraction.')


# =====================================================================
# 1. LOAD DATA
# =====================================================================
print('=' * 70)
print('STEP 1: Load data')
print('=' * 70)

# Explicitly set DATA_DIR again right before use to ensure correct path
DATA_DIR = '/content/gayanara-forecasting/data/'
print(f"Loading data from: {DATA_DIR}")

customers = pd.read_csv(DATA_DIR + 'customers.csv')
orders = pd.read_csv(DATA_DIR + 'orders.csv', parse_dates=['order_date'])
items = pd.read_csv(DATA_DIR + 'order_items.csv')
products = pd.read_csv(DATA_DIR + 'products.csv')
reviews = pd.read_csv(DATA_DIR + 'reviews.csv', parse_dates=['review_date'])

print(f'customers   : {customers.shape}')
print(f'orders      : {orders.shape}  ({orders.order_date.min().date()} - {orders.order_date.max().date()})')
print(f'order_items : {items.shape}')
print(f'products    : {products.shape}')
print(f'reviews     : {reviews.shape}')

# =====================================================================
# 2. DATA CLEANING
#    Kolom 'category' di products.csv mencampur label EN/ID dan huruf
#    besar-kecil yang tidak konsisten (mis. 'Dress'/'dress',
#    'Jacket'/'Jaket', 'Accessories'/'Aksesoris'). Distandardisasi agar
#    KPI per kategori tidak terpecah ke kategori duplikat.
# =====================================================================
print('\n' + '=' * 70)
print('STEP 2: Data cleaning — standardisasi kategori produk')
print('=' * 70)

category_map = {
    'dress': 'Dress', 'Dress': 'Dress',
    'jacket': 'Jacket', 'Jacket': 'Jacket', 'jaket': 'Jacket', 'Jaket': 'Jacket',
    'accessories': 'Accessories', 'Accessories': 'Accessories',
    'aksesoris': 'Accessories', 'Aksesoris': 'Accessories',
    'celana': 'Pants', 'Celana': 'Pants', 'pants': 'Pants', 'Pants': 'Pants',
    'kaos': 'T-Shirt', 'Kaos': 'T-Shirt', 't-shirt': 'T-Shirt', 'T-Shirt': 'T-Shirt',
    'kemeja': 'Shirt', 'Kemeja': 'Shirt', 'shirt': 'Shirt', 'Shirt': 'Shirt',
}
products['category_raw'] = products['category']
products['category'] = products['category'].map(category_map).fillna(products['category'])
print('Sebelum :', sorted(products['category_raw'].unique()))
print('Sesudah :', sorted(products['category'].unique()))

# =====================================================================
# 3. BUILD FACT TABLE (order_items + orders + products + customers)
# =====================================================================
print('\n' + '=' * 70)
print('STEP 3: Build fact table')
print('=' * 70)

fact = (items
        .merge(orders, on='order_id', how='left')
        .merge(products[['product_id', 'category', 'sub_category', 'brand', 'material']], on='product_id', how='left')
        .merge(customers[['customer_id', 'gender', 'age_group', 'province', 'city']],
               on='customer_id', how='left'))

fact['order_date'] = pd.to_datetime(fact['order_date'])
fact['order_month'] = fact['order_date'].dt.to_period('M').dt.to_timestamp()
fact['is_cancelled'] = fact['order_status'].eq('cancelled')
fact['net_valid'] = ~fact['is_cancelled']
fact_valid = fact[fact['net_valid']].copy()

fact.to_csv(OUT_DIR + 'fact_order_items.csv', index=False)
print(f'fact table: {fact.shape}')

# Revenue diakui dari orders.total_amount_idr, MENGECUALIKAN status 'cancelled'.
# Status 'returned' tetap dihitung sebagai revenue historis tapi dilacak
# terpisah lewat KPI Return Rate.
orders_valid = orders[orders['order_status'] != 'cancelled'].copy()
orders_all = orders.copy()

# =====================================================================
# 4. KPI UTAMA
# =====================================================================
print('\n' + '=' * 70)
print('STEP 4: Hitung KPI utama')
print('=' * 70)

total_revenue = orders_valid['total_amount_idr'].sum()
total_orders_valid = orders_valid.shape[0]
total_orders_all = orders_all.shape[0]
aov = total_revenue / total_orders_valid
orders_per_cust = orders_valid.groupby('customer_id').size()
total_customers = orders_per_cust.shape[0]  # pelanggan dengan >=1 order valid

cancel_rate = (orders_all['order_status'] == 'cancelled').mean()
return_rate = (orders_all['order_status'] == 'returned').mean()

repeat_rate = (orders_per_cust >= 2).mean()

avg_rating = reviews['rating'].mean()

kpi_summary = pd.DataFrame([
    {'KPI': 'Total Revenue (IDR, net of cancelled)', 'Value': total_revenue},
    {'KPI': 'Total Orders (valid)', 'Value': total_orders_valid},
    {'KPI': 'Total Orders (all incl. cancelled)', 'Value': total_orders_all},
    {'KPI': 'Average Order Value (AOV, IDR)', 'Value': aov},
    {'KPI': 'Total Customers', 'Value': total_customers},
    {'KPI': 'Cancellation Rate', 'Value': cancel_rate},
    {'KPI': 'Return Rate', 'Value': return_rate},
    {'KPI': 'Repeat Customer Rate (>=2 valid orders)', 'Value': repeat_rate},
    {'KPI': 'Average Product Rating', 'Value': avg_rating},
])
kpi_summary.to_csv(OUT_DIR + 'kpi_summary.csv', index=False)
kpi_dict = dict(zip(kpi_summary['KPI'], kpi_summary['Value']))
print(kpi_summary.to_string(index=False))
print(f"(Catatan: {customers.shape[0]} pelanggan terdaftar total; {total_customers} di antaranya punya >=1 order valid)")

# =====================================================================
# 5. MONTHLY REVENUE (basis forecasting)
# =====================================================================
monthly = (orders_valid
           .assign(order_month=orders_valid['order_date'].dt.to_period('M').dt.to_timestamp())
           .groupby('order_month')
           .agg(revenue_idr=('total_amount_idr', 'sum'), orders=('order_id', 'nunique'))
           .reset_index())
monthly['aov_idr'] = monthly['revenue_idr'] / monthly['orders']
monthly.to_csv(OUT_DIR + 'monthly_revenue.csv', index=False)

# =====================================================================
# 6. BREAKDOWN PER DIMENSI (kategori, provinsi, pembayaran, dll.)
# =====================================================================
print('\n' + '=' * 70)
print('STEP 5-6: Monthly revenue & breakdown per dimensi')
print('=' * 70)

by_category = (fact_valid.groupby('category')
               .agg(revenue_idr=('subtotal_idr', 'sum'), qty=('quantity', 'sum'), items_sold=('item_id', 'count'))
               .reset_index().sort_values('revenue_idr', ascending=False))
by_category.to_csv(OUT_DIR + 'sales_by_category.csv', index=False)

by_subcategory = (fact_valid.groupby(['category', 'sub_category'])
                   .agg(revenue_idr=('subtotal_idr', 'sum'), qty=('quantity', 'sum'))
                   .reset_index().sort_values('revenue_idr', ascending=False))
by_subcategory.to_csv(OUT_DIR + 'sales_by_subcategory.csv', index=False)

by_province = (orders_valid.groupby('shipping_province')
               .agg(revenue_idr=('total_amount_idr', 'sum'), orders=('order_id', 'nunique'))
               .reset_index().sort_values('revenue_idr', ascending=False))
by_province.to_csv(OUT_DIR + 'sales_by_province.csv', index=False)

by_payment = (orders_valid.groupby('payment_method')
              .agg(revenue_idr=('total_amount_idr', 'sum'), orders=('order_id', 'nunique'))
              .reset_index().sort_values('revenue_idr', ascending=False))
by_payment.to_csv(OUT_DIR + 'sales_by_payment.csv', index=False)

by_courier = (orders_valid.groupby('courier')
              .agg(orders=('order_id', 'nunique'), avg_shipping_cost=('shipping_cost_idr', 'mean'))
              .reset_index())
by_courier.to_csv(OUT_DIR + 'orders_by_courier.csv', index=False)

top_products = (fact_valid.groupby('product_id')
                .agg(revenue_idr=('subtotal_idr', 'sum'), qty=('quantity', 'sum'))
                .reset_index()
                .merge(products[['product_id', 'name', 'category', 'brand', 'avg_rating']], on='product_id')
                .sort_values('revenue_idr', ascending=False).head(20))
top_products.to_csv(OUT_DIR + 'top_products.csv', index=False)

by_segment = (fact_valid.groupby(['gender', 'age_group'])
              .agg(revenue_idr=('subtotal_idr', 'sum'), customers=('customer_id', 'nunique'))
              .reset_index().sort_values('revenue_idr', ascending=False))
by_segment.to_csv(OUT_DIR + 'sales_by_segment.csv', index=False)

status_breakdown = orders_all['order_status'].value_counts().reset_index()
status_breakdown.columns = ['order_status', 'count']
status_breakdown.to_csv(OUT_DIR + 'order_status_breakdown.csv', index=False)

cust_orders = orders_valid.groupby('customer_id').size().reset_index(name='valid_order_count')
cust_orders.to_csv(OUT_DIR + 'customer_order_counts.csv', index=False)

print('Top kategori:\n', by_category.head(3).to_string(index=False))
print('\nTop provinsi:\n', by_province.head(3).to_string(index=False))

# =====================================================================
# 7. FORECASTING — Holt-Winters Exponential Smoothing
# =====================================================================
print('\n' + '=' * 70)
print('STEP 7: Forecasting revenue (Holt-Winters)')
print('=' * 70)

series = monthly.set_index('order_month')['revenue_idr'].asfreq('MS')

model = ExponentialSmoothing(
    series, trend='add', seasonal='add', seasonal_periods=12,
    initialization_method='estimated'
).fit(optimized=True)

forecast = model.forecast(FORECAST_HORIZON)

resid_std = np.std(model.resid)
z80, z95 = 1.2816, 1.96
horizon_idx = np.arange(1, FORECAST_HORIZON + 1)
widen = np.sqrt(horizon_idx)

fc_df = pd.DataFrame({
    'order_month': forecast.index,
    'forecast_revenue_idr': forecast.values,
    'lower_80': (forecast.values - z80 * resid_std * widen).clip(min=0),
    'upper_80': forecast.values + z80 * resid_std * widen,
    'lower_95': (forecast.values - z95 * resid_std * widen).clip(min=0),
    'upper_95': forecast.values + z95 * resid_std * widen,
})
fc_df.to_csv(OUT_DIR + 'revenue_forecast.csv', index=False)

# Gabungan actual + forecast (untuk chart / dashboard)
hist_long = pd.DataFrame({
    'order_month': series.index, 'revenue_idr': series.values,
    'series_type': 'Actual', 'lower_95': np.nan, 'upper_95': np.nan,
})
fc_long = pd.DataFrame({
    'order_month': fc_df['order_month'], 'revenue_idr': fc_df['forecast_revenue_idr'],
    'series_type': 'Forecast', 'lower_95': fc_df['lower_95'], 'upper_95': fc_df['upper_95'],
})
monthly_fc = pd.concat([hist_long, fc_long], ignore_index=True)
monthly_fc.to_csv(OUT_DIR + 'revenue_actual_vs_forecast.csv', index=False)

# Back-test: fit tanpa 6 bulan terakhir, forecast, bandingkan dengan aktual
train, test = series.iloc[:-6], series.iloc[-6:]
bt_model = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=12,
                                 initialization_method='estimated').fit(optimized=True)
bt_forecast = bt_model.forecast(6)
mape = float(np.mean(np.abs((test.values - bt_forecast.values) / test.values)) * 100)

print(f'Forecast {FORECAST_HORIZON} bulan ke depan:')
print(fc_df[['order_month', 'forecast_revenue_idr', 'lower_95', 'upper_95']].round(0).to_string(index=False))
print(f'\nBack-test MAPE (6 bulan terakhir): {mape:.1f}%')

# =====================================================================
# 8. VISUALISASI — Dashboard KPI (gambar utama)
# =====================================================================
print('\n' + '=' * 70)
print('STEP 8: Membuat visualisasi')
print('=' * 70)

actual = monthly_fc[monthly_fc['series_type'] == 'Actual']
fc_plot = monthly_fc[monthly_fc['series_type'] == 'Forecast']
cat = by_category.sort_values('revenue_idr', ascending=False)
prov10 = by_province.sort_values('revenue_idr', ascending=False).head(10)
pay = by_payment.sort_values('revenue_idr', ascending=False)

fig = plt.figure(figsize=(14, 9))
gs = GridSpec(4, 3, figure=fig, height_ratios=[0.55, 0.9, 1.3, 1.3], hspace=0.55, wspace=0.35)
fig.suptitle('GAYANARA — Sales Performance & Forecasting Dashboard', fontsize=19, fontweight='bold', color=DARK, y=1.015)
fig.text(0.5, 0.975, 'E-commerce Fashion Analytics  |  Periode data: Jan 2022 – Feb 2025', ha='center', fontsize=11, color=GREY)

kpi_cards = [
    ('Total Revenue', fmt_idr(kpi_dict['Total Revenue (IDR, net of cancelled)'])),
    ('Total Orders', f"{int(kpi_dict['Total Orders (valid)']):,}".replace(',', '.')),
    ('AOV', fmt_idr(kpi_dict['Average Order Value (AOV, IDR)'])),
    ('Repeat Rate', f"{kpi_dict['Repeat Customer Rate (>=2 valid orders)']*100:.1f}%"),
    ('Avg Rating', f"{kpi_dict['Average Product Rating']:.2f} / 5"),
    ('Cancel Rate', f"{kpi_dict['Cancellation Rate']*100:.1f}%"),
]
for i, (label, val) in enumerate(kpi_cards):
    ax = fig.add_subplot(gs[0, :])
    ax.axis('off')
    x0, w = i / 6, 1 / 6 - 0.012
    rect = FancyBboxPatch((x0, 0), w, 1, boxstyle="round,pad=0.005,rounding_size=0.02",
                           transform=ax.transAxes, linewidth=1.2, edgecolor=ROSE_GOLD, facecolor=LIGHT_PINK, clip_on=False)
    ax.add_patch(rect)
    ax.text(x0 + w/2, 0.62, val, transform=ax.transAxes, ha='center', va='center', fontsize=13.5, fontweight='bold', color=DARK)
    ax.text(x0 + w/2, 0.22, label, transform=ax.transAxes, ha='center', va='center', fontsize=9.5, color=GREY)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

ax1 = fig.add_subplot(gs[1:3, :2])
ax1.plot(actual['order_month'], actual['revenue_idr'] / 1e6, color=DARK, linewidth=2, label='Actual')
ax1.plot(fc_plot['order_month'], fc_plot['revenue_idr'] / 1e6, color=ROSE_GOLD, linewidth=2, linestyle='--', marker='o', markersize=4, label='Forecast (6 bulan)')
ax1.fill_between(fc_plot['order_month'], fc_plot['lower_95'] / 1e6, fc_plot['upper_95'] / 1e6, color=ROSE_GOLD, alpha=0.18, label='95% Prediction Interval')
ax1.plot([actual['order_month'].iloc[-1], fc_plot['order_month'].iloc[0]],
         [actual['revenue_idr'].iloc[-1] / 1e6, fc_plot['revenue_idr'].iloc[0] / 1e6], color=ROSE_GOLD, linewidth=2, linestyle='--')
ax1.set_title('Revenue Bulanan: Actual vs Forecast', fontsize=12.5, fontweight='bold', color=DARK, loc='left')
ax1.set_ylabel('Revenue (Juta IDR)', fontsize=9.5)
ax1.legend(loc='upper left', frameon=False, fontsize=8.5)
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
plt.setp(ax1.get_xticklabels(), rotation=40, ha='right', fontsize=8.5)
style_ax(ax1)

ax2 = fig.add_subplot(gs[1, 2])
colors2 = [MINT if i % 2 == 0 else ROSE_GOLD for i in range(len(cat))]
ax2.barh(cat['category'][::-1], cat['revenue_idr'][::-1] / 1e6, color=colors2[::-1])
ax2.set_title('Revenue per Kategori (Jt IDR)', fontsize=10.5, fontweight='bold', color=DARK, loc='left')
ax2.tick_params(labelsize=8.5)
style_ax(ax2)

ax3 = fig.add_subplot(gs[2, 2])
pay_colors = [ROSE_GOLD, MINT, DARK, '#D9A9AF', '#8FCBAF'][:len(pay)]
wedges, _, autotexts = ax3.pie(pay['revenue_idr'], colors=pay_colors, autopct='%1.0f%%', pctdistance=0.78,
                                 startangle=90, wedgeprops={'width': 0.42, 'edgecolor': 'white'})
plt.setp(autotexts, size=8, color='white', fontweight='bold')
ax3.legend(pay['payment_method'], loc='upper center', fontsize=7.5, frameon=False, bbox_to_anchor=(0.5, -0.05), ncol=2)
ax3.set_title('Metode Pembayaran (Revenue)', fontsize=10.5, fontweight='bold', color=DARK, loc='left', pad=12)

ax4 = fig.add_subplot(gs[3, :])
colors4 = [DARK if i == 0 else (MINT if i % 2 == 0 else ROSE_GOLD) for i in range(len(prov10))]
ax4.bar(prov10['shipping_province'], prov10['revenue_idr'] / 1e6, color=colors4)
ax4.set_title('Top 10 Provinsi Berdasarkan Revenue (Jt IDR)', fontsize=11, fontweight='bold', color=DARK, loc='left')
ax4.tick_params(axis='x', labelrotation=30, labelsize=8)
for tick in ax4.get_xticklabels():
    tick.set_ha('right')
style_ax(ax4)

fig.text(0.5, 0.005, 'Sumber: orders.csv, order_items.csv, products.csv, customers.csv, reviews.csv  |  Python (pandas, matplotlib, statsmodels)',
          ha='center', fontsize=8, color='#AAAAAA')
plt.savefig(VIZ_DIR + '01_kpi_dashboard.png', dpi=160, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved: visuals/01_kpi_dashboard.png')

# =====================================================================
# 9. VISUALISASI — chart pendukung individual
# =====================================================================

# 02: Revenue forecast (standalone)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(actual['order_month'], actual['revenue_idr'] / 1e6, color=DARK, linewidth=2.2, label='Actual')
ax.plot(fc_plot['order_month'], fc_plot['revenue_idr'] / 1e6, color=ROSE_GOLD, linewidth=2.2, linestyle='--', marker='o', markersize=5, label='Forecast (6 bulan)')
ax.fill_between(fc_plot['order_month'], fc_plot['lower_95'] / 1e6, fc_plot['upper_95'] / 1e6, color=ROSE_GOLD, alpha=0.18, label='95% Prediction Interval')
ax.plot([actual['order_month'].iloc[-1], fc_plot['order_month'].iloc[0]],
        [actual['revenue_idr'].iloc[-1] / 1e6, fc_plot['revenue_idr'].iloc[0] / 1e6], color=ROSE_GOLD, linewidth=2.2, linestyle='--')
ax.set_title('Revenue Bulanan: Actual vs Forecast (Holt-Winters Exponential Smoothing)', fontsize=13, fontweight='bold', color=DARK, loc='left', pad=12)
ax.set_ylabel('Revenue (Juta IDR)')
ax.legend(loc='upper left', frameon=False)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45, ha='right')
style_ax(ax)
plt.tight_layout()
plt.savefig(VIZ_DIR + '02_revenue_forecast.png', dpi=160, facecolor='white')
plt.close()

# 03: Category performance
cat_sorted = cat.sort_values('revenue_idr')
fig, ax = plt.subplots(figsize=(9, 4.5))
colors = [MINT if i % 2 == 0 else ROSE_GOLD for i in range(len(cat_sorted))]
ax.barh(cat_sorted['category'], cat_sorted['revenue_idr'] / 1e6, color=colors)
ax.set_title('Revenue per Kategori Produk (Juta IDR)', fontsize=13, fontweight='bold', color=DARK, loc='left')
for i, v in enumerate(cat_sorted['revenue_idr'] / 1e6):
    ax.text(v + 3, i, f'{v:,.0f}', va='center', fontsize=9, color=DARK)
style_ax(ax)
ax.grid(axis='x', alpha=0.25)
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.savefig(VIZ_DIR + '03_category_performance.png', dpi=160, facecolor='white')
plt.close()

# 04: Province performance
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(prov10['shipping_province'], prov10['revenue_idr'] / 1e6, color=colors4)
ax.set_title('Top 10 Provinsi Berdasarkan Revenue (Juta IDR)', fontsize=13, fontweight='bold', color=DARK, loc='left')
ax.set_ylabel('Revenue (Juta IDR)')
plt.xticks(rotation=30, ha='right')
style_ax(ax)
plt.tight_layout()
plt.savefig(VIZ_DIR + '04_province_performance.png', dpi=160, facecolor='white')
plt.close()

# 05: Payment method donut
fig, ax = plt.subplots(figsize=(6.5, 6.5))
wedges, _, autotexts = ax.pie(pay['revenue_idr'], colors=pay_colors, autopct='%1.0f%%', pctdistance=0.78,
                                startangle=90, wedgeprops={'width': 0.42, 'edgecolor': 'white'})
plt.setp(autotexts, size=10, color='white', fontweight='bold')
ax.legend(pay['payment_method'], loc='upper center', fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.02), ncol=3)
ax.set_title('Distribusi Revenue per Metode Pembayaran', fontsize=13, fontweight='bold', color=DARK, pad=15)
plt.tight_layout()
plt.savefig(VIZ_DIR + '05_payment_method.png', dpi=160, facecolor='white')
plt.close()

# 06: Order status breakdown
status_sorted = status_breakdown.sort_values('count', ascending=False)
status_colors_map = {'delivered': MINT, 'shipped': '#8FCBAF', 'processing': '#D9A9AF', 'cancelled': ROSE_GOLD, 'returned': DARK}
fig, ax = plt.subplots(figsize=(8, 4.5))
colors6 = [status_colors_map.get(s, GREY) for s in status_sorted['order_status']]
bars = ax.bar(status_sorted['order_status'], status_sorted['count'], color=colors6)
ax.set_title('Distribusi Order Berdasarkan Status', fontsize=13, fontweight='bold', color=DARK, loc='left')
ax.set_ylabel('Jumlah Order')
for b in bars:
    h = b.get_height()
    ax.text(b.get_x() + b.get_width()/2, h + 15, f'{int(h):,}', ha='center', fontsize=9.5, color=DARK)
style_ax(ax)
plt.tight_layout()
plt.savefig(VIZ_DIR + '06_order_status.png', dpi=160, facecolor='white')
plt.close()

# 07: Customer segment heatmap (gender x age_group)
pivot = by_segment.pivot(index='gender', columns='age_group', values='revenue_idr') / 1e6
age_order = ['18-24', '25-34', '35-44', '45-54', '55+']
age_order = [a for a in age_order if a in pivot.columns] + [c for c in pivot.columns if c not in age_order]
pivot = pivot[age_order]
fig, ax = plt.subplots(figsize=(9, 3.6))
im = ax.imshow(pivot.values, cmap='RdPu', aspect='auto')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        color = 'white' if val > pivot.values.max() * 0.55 else DARK
        ax.text(j, i, f'{val:,.0f}', ha='center', va='center', color=color, fontsize=10, fontweight='bold')
ax.set_title('Revenue per Segmen Pelanggan (Juta IDR): Gender x Kelompok Usia', fontsize=12.5, fontweight='bold', color=DARK, loc='left', pad=12)
plt.tight_layout()
plt.savefig(VIZ_DIR + '07_customer_segment.png', dpi=160, facecolor='white')
plt.close()

# 08: Top 10 products
topp = top_products.sort_values('revenue_idr', ascending=False).head(10).sort_values('revenue_idr')
fig, ax = plt.subplots(figsize=(10, 5.5))
colors8 = [MINT if i % 2 == 0 else ROSE_GOLD for i in range(len(topp))]
labels = [f"{n[:28]}{'…' if len(n) > 28 else ''}" for n in topp['name']]
ax.barh(labels, topp['revenue_idr'] / 1e6, color=colors8)
ax.set_title('Top 10 Produk Berdasarkan Revenue (Juta IDR)', fontsize=13, fontweight='bold', color=DARK, loc='left')
for i, v in enumerate(topp['revenue_idr'] / 1e6):
    ax.text(v + 0.3, i, f'{v:,.1f}', va='center', fontsize=9, color=DARK)
style_ax(ax)
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.savefig(VIZ_DIR + '08_top_products.png', dpi=160, facecolor='white')
plt.close()

print('Saved: visuals/02..08_*.png')

# =====================================================================
# 10. INSIGHT & REKOMENDASI BISNIS (ringkasan otomatis)
# =====================================================================
print('\n' + '=' * 70)
print('STEP 9: KPI & INSIGHT SUMMARY')
print('=' * 70)

top_cat = cat.iloc[0]
top_prov = by_province.iloc[0]
top_pay = pay.iloc[0]
next_month_fc = fc_df.iloc[0]

print(f"""
KPI UTAMA
---------
- Total Revenue (net)         : {fmt_idr(total_revenue, short=False)}
- Total Valid Orders           : {total_orders_valid:,}
- Average Order Value (AOV)    : {fmt_idr(aov, short=False)}
- Total Customers               : {total_customers:,}
- Cancellation Rate             : {cancel_rate*100:.1f}%
- Return Rate                   : {return_rate*100:.1f}%
- Repeat Customer Rate          : {repeat_rate*100:.1f}%
- Average Product Rating        : {avg_rating:.2f} / 5

FORECAST
--------
- Model                         : Holt-Winters Exponential Smoothing (trend + seasonal aditif, siklus 12 bulan)
- Horizon                       : {FORECAST_HORIZON} bulan ke depan
- Back-test MAPE                : {mape:.1f}%
- Forecast bulan berikutnya     : {fmt_idr(next_month_fc['forecast_revenue_idr'], short=False)}
  (interval 95%: {fmt_idr(next_month_fc['lower_95'], short=False)} - {fmt_idr(next_month_fc['upper_95'], short=False)})

INSIGHT & REKOMENDASI
----------------------
1. Revenue tumbuh konsisten YoY (2022->2023 ~2x, 2023->2024 ~15%), dengan lonjakan
   tajam di Jan-Feb 2025 — perlu dikonfirmasi apakah didorong promo/musiman.
2. Kategori '{top_cat['category']}' adalah penyumbang revenue terbesar
   ({fmt_idr(top_cat['revenue_idr'], short=False)}), namun 3 kategori teratas (Jacket, Pants, Dress)
   berkontribusi hampir merata — peluang bundling promo lintas kategori.
3. Cancellation Rate {cancel_rate*100:.1f}% cukup material — disarankan investigasi lanjutan
   pada payment_method / courier dengan cancel rate tertinggi.
4. Provinsi '{top_prov['shipping_province']}' mendominasi revenue regional
   ({fmt_idr(top_prov['revenue_idr'], short=False)}) — relevan untuk keputusan lokasi gudang/ekspansi kurir.
5. Metode pembayaran '{top_pay['payment_method']}' paling banyak digunakan — pastikan
   reliabilitas payment gateway ini tetap terjaga.
6. Forecast 6 bulan menunjukkan revenue stabil di kisaran Rp 59-72 juta/bulan,
   baik sebagai dasar target penjualan kuartal berikutnya.

Semua tabel hasil tersimpan di: {OUT_DIR}
Semua chart tersimpan di      : {VIZ_DIR}
""")

print('=' * 70)
print('SELESAI')
print('=' * 70)

Data directory already exists, skipping extraction.
STEP 1: Load data
Loading data from: /content/gayanara-forecasting/data/
customers   : (800, 9)
orders      : (3000, 12)  (2022-01-01 - 2025-02-28)
order_items : (4986, 6)
products    : (300, 9)
reviews     : (1500, 8)

STEP 2: Data cleaning — standardisasi kategori produk
Sebelum : ['Accessories', 'Aksesoris', 'Celana', 'Dress', 'Jacket', 'Jaket', 'Kaos', 'Kemeja', 'Pants', 'Shirt', 'T-Shirt', 'dress', 'kaos', 'kemeja']
Sesudah : ['Accessories', 'Dress', 'Jacket', 'Pants', 'Shirt', 'T-Shirt']

STEP 3: Build fact table
fact table: (4986, 28)

STEP 4: Hitung KPI utama
                                    KPI        Value
  Total Revenue (IDR, net of cancelled) 1.345149e+09
                   Total Orders (valid) 2.672000e+03
     Total Orders (all incl. cancelled) 3.000000e+03
         Average Order Value (AOV, IDR) 5.034239e+05
                        Total Customers 7.750000e+02
                      Cancellation Rate 1.093333e-01
   